In [1]:
import gymnasium as gym
from utils_v2 import KalmanFilter
import jax.numpy as jnp
import numpy as np
from gym.spaces import Box
from math import sqrt

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
env = gym.make("CartPole-v1")


In [ ]:
class RlKf(gym.Env):
    def __init__(
        self,
        kf: KalmanFilter,
        threshold: float = 0.00001,
        max_episodes=100,
    ):
        super(RlKf, self).__init__()
        self.x_0 = kf.x_0

        state = Box(
            low=np.ones_like(self.x_0) * self.x_0.min().item(),
            high=np.ones_like(self.x_0) * self.x_0.max().item(),
        )
        self.action_space = state
        self.observation_space = state
        self.kf_0 = kf
        self.kf = kf
        self.max_episodes = max_episodes
        self.episodes = 0
        self.threshold = threshold

    def reset(self, *, seed=None, options=None):
        self.kf = self.kf_0
        return self.x_0, {}

    def step(self, action):
        self.episodes += 1
        self.kf.predict(action)
        self.kf.update(self.x_0)

        reward = jnp.sqrt(jnp.mean((self.kf.x_k - self.x_0) ** 2))
        done = True if reward < self.threshold else False
        truncuated = True if self.episodes > self.max_episodes else False
        return self.kf.x_k, -reward, done, truncuated, {}
